# 기획표 → Final Cut Pro FCPXML · Colab Safe Quick v0.6.0

이 노트북은 **기획표 1개(`.csv` 또는 `.xlsx`)와 사진·영상 파일**을 직접 업로드하여 Final Cut Pro용 러프컷을 만듭니다. 사용자는 사진인지 영상인지 `종류`를 적거나 최종 타임라인 시간을 계산할 필요가 없습니다. 기본적으로 기획표에 적은 행 순서대로 이어 붙이고, 미디어 종류와 길이는 자동으로 검사합니다.

## 실행 전에 꼭 확인하세요

- 최신 Chrome, Edge 또는 Safari가 설치된 Windows·macOS·Linux의 브라우저에서 XML을 만들 수 있습니다.
- **생성된 FCPXML을 실제로 가져오고 확인하는 단계는 Final Cut Pro가 설치된 Mac에서만 가능합니다.** Colab은 Final Cut Pro를 실행하지 않습니다.
- Final Cut Pro가 있는 개인 Mac에서는 로컬 가상환경 실행을 권장합니다. 이 Colab은 설치 없이 먼저 시험하거나 다른 운영체제에서 기획표를 변환할 때 쓰는 선택 경로입니다.
- 파일은 이 노트북의 Google Colab 임시 런타임으로 직접 올라갑니다. 회사 기밀, NDA 원본, 민감한 얼굴·음성은 업로드 정책을 먼저 확인하세요.
- 이 노트북은 개인 저장소를 연결하지 않고 공개 공유 서버도 열지 않습니다. 작업 후 마지막 안내대로 런타임을 삭제하세요.

## 전체 순서

1. 아래 설정에서 세로 9:16, 가로 16:9 또는 둘 다를 선택합니다.
2. 타임라인 기획표(`.csv` 또는 `.xlsx`)를 **정확히 1개** 올립니다.
3. 정밀 자막을 쓸 때만 자막 기획표(`.csv` 또는 `.xlsx`)를 **정확히 1개** 올립니다.
4. 표시된 표에서 파일명·시간·자막 문구를 확인하고 `OK`를 입력합니다.
5. 기획표에 적은 사진·영상 파일을 한 번에 여러 개 올립니다.
6. 입력 검사를 통과하면 FCPXML을 만듭니다. `both`를 고르면 한 번 올린 파일로 세로와 가로를 순서대로 생성합니다.
7. 방향별 폴더로 분리된 허용 결과만 하나의 ZIP으로 내려받습니다.


## 0. 기획표를 이렇게 작성합니다

처음에는 아래 **6개 열**만 사용하면 됩니다. Excel은 `.xlsx` 그대로 올릴 수 있고, CSV는 `CSV UTF-8`로 저장합니다. Numbers는 `파일 → 다음으로 내보내기 → Excel`로 `.xlsx`를 만든 뒤 올리세요.

같은 장면 순서·원본 구간·자막을 세로와 가로에 공통으로 쓴다면 기획표도 미디어도 한 번만 올리면 됩니다. `portrait`는 세로 9:16(1080×1920), `landscape`는 가로 16:9(1920×1080), `both`는 두 결과를 모두 만듭니다. 플랫폼마다 장면 순서나 길이가 다르면 기획표를 각각 만들어 별도 실행하세요.

```csv
파일,영상 원본 시작,영상 원본 끝,사진 표시 시간(초),화면 자막,소리
cafe_boot.mov,00:00:12.000,00:00:16.000,,카페에서 첫 부팅,사용
ipad_drawing.png,,,3,아이패드로 그린 그림,
office.mp4,,,,회사에서도 사용,끄기
```

| 열 | 사용자가 지정하는 값 | 비워 두면 |
|---|---|---|
| `파일` | 업로드할 실제 파일명과 확장자 | 오류로 안내 |
| `영상 원본 시작` | 원본 영상에서 가져오기 시작할 시점 | 시작·끝을 모두 비우면 영상 전체 |
| `영상 원본 끝` | 원본 영상에서 가져오기를 끝낼 시점 | 시작·끝을 모두 비우면 영상 전체 |
| `사진 표시 시간(초)` | 사진을 보여줄 초: `2.5`, `3`, `5` | 위 설정의 기본 사진 시간 |
| `화면 자막` | 해당 장면 위에 표시할 일반 자막 | 자막 없음 |
| `소리` | 영상은 `사용` 또는 `끄기` | 영상은 원본 소리 사용, 사진은 무시 |

기획표에 적은 행 순서가 곧 편집 순서입니다. 순서를 별도로 관리하거나 화면 맞춤·메모가 필요하면 `순서`, `화면 맞춤`, `메모` 열을 선택해서 추가할 수 있습니다. `순서` 열을 추가했다면 모든 장면에 중복 없는 1 이상의 정수를 적으세요.

### 가장 헷갈리는 시간의 의미

`영상 원본 시작=00:00:12.000`, `영상 원본 끝=00:00:16.000`은 **완성 영상의 12~16초가 아니라, 업로드한 원본 영상에서 12초 지점부터 16초 지점까지 4초를 사용한다**는 뜻입니다. 사진에는 원본 시간이 없으므로 두 칸을 비우고 `사진 표시 시간(초)`만 적습니다. 최종 타임라인의 시작·끝은 앞 장면부터 길이를 누적하여 자동 계산합니다.

### 화면 맞춤 값

- 빈칸 또는 `전체 보이기` → `fit`: 사진·영상을 자르지 않고 전체 표시
- `화면 채우기` → `fill`: 화면을 꽉 채우며 가장자리가 잘릴 수 있음
- `자동 맞춤 안 함` → `none`: 자동 크기 맞춤을 사용하지 않는 고급 선택

### 파일명 규칙

- 기획표에는 폴더 경로가 아닌 `cafe_boot.mov` 같은 **파일명만** 씁니다.
- 기획표의 파일명은 업로드한 이름과 확장자·대소문자까지 같아야 합니다. `Intro.MOV`와 `intro.mov`는 다른 이름입니다.
- 이름이 대소문자 또는 Unicode 표기만 달라 사실상 같은 파일이 두 개면 중단합니다.
- 같은 미디어를 여러 행에서 다시 사용하는 것은 가능하지만, 업로드 파일명 자체는 중복되면 안 됩니다.
- 지원 기본 형식: 영상 `.mp4 .mov .m4v .avi .mkv .webm`, 사진 `.jpg .jpeg .png .tif .tiff .heic .webp .bmp`

### 한 장면에 자막이 여러 번 바뀔 때만 정밀 자막 기획표 사용

설정에서 `USE_SEPARATE_SUBTITLES_CSV`를 켜고 아래 5개 열의 `.csv` 또는 `.xlsx` 기획표를 올립니다. `시작`과 `끝`은 **완성 영상 전체 타임라인 기준**입니다. 별도 자막 기획표를 쓰면 타임라인 기획표의 `화면 자막`은 비워 두는 것이 가장 명확합니다.

```csv
번호,시작,끝,최종 대사,장면 의도
S01,00:00:00.200,00:00:01.800,드디어 맥북을 켰다,오프닝
S02,00:00:02.000,00:00:03.800,그런데 생각보다 어렵다,반전
```


In [ ]:
# 1. 사용자 설정 — 오른쪽 입력칸만 바꾸고 코드는 수정하지 않아도 됩니다.
PROJECT_NAME = "MyVideo"  # @param {type:"string"}
# portrait=세로 9:16 · landscape=가로 16:9 · both=세로+가로 둘 다
OUTPUT_FORMAT = "portrait"  # @param ["portrait", "landscape", "both"]
FPS = "30"  # @param ["23.976", "24", "25", "29.97", "30", "50", "59.94", "60"]
SUBTITLE_MODE = "title"  # @param ["title", "caption", "both", "off"]
EXPORT_SRT = True  # @param {type:"boolean"}
DEFAULT_PHOTO_DURATION = 3.0  # @param {type:"number"}
USE_SEPARATE_SUBTITLES_CSV = False  # @param {type:"boolean"}


In [ ]:
# 개발자 전용 배포 고정값입니다. 일반 사용자는 이 셀을 펼치거나 수정하지 않습니다.
# 게시자는 main 대신 검토한 릴리스 태그 또는 고정 commit ref를 사용하세요.
REPO_URL = "https://github.com/Kongdataif/csv-to-fcpxml-starter.git"
REPO_REF = "v0.6.0"

# v0.6.0에서 검증한 실행 소스 일곱 파일의 SHA-256입니다. 코드를 바꾼 새 릴리스는 이 값도 갱신합니다.
# public GitHub 모드에서는 이 값이 준비되지 않았거나 하나라도 다르면 실행을 중단합니다.
SOURCE_HASHES_READY = True
SOURCE_SHA256 = {
    "make_xml.py": "d78f67f94387dedaaf31a92ab0e842deee983efd1754afa7a4939e8abcc9a174",
    "make_xml_input.py": "6b994612584b1518414a365d07fc867d6964c73c9945d5fc312e5c075749cffe",
    "csv_to_fcpxml.py": "3334a06a1c435ae244ac048d277d8f9df8a679358dc60a9c161f1b1b09837fb5",
    "simple_timeline.py": "339bba47912b2a87fe3d29e4a7638ad788ff2427c4b79de1107c7d9a2d6ec109",
    "spreadsheet_input.py": "03599e35ce43db9f5bba74901169ece1ba85f1b18629c3646761d9847866a02f",
    "preview_report.py": "a48a6f9d42bd5109fc9a6417baf604be588785cd774c52bbd30a3c7c378d6500",
    "scripts/validate_fcpxml.py": "729c71e36a9b91bdcce5b3ab3eb6f4ea79f7fab71d5aabf63a6e220fdab43856",
}


## 1. 변환 코드와 안전한 임시 작업공간 준비

이 단계는 배포자가 고정한 공개 릴리스 코드를 확인하고 안전한 임시 작업공간을 준비합니다. 일반 사용자는 값을 바꾸지 않고 셀을 실행하면 됩니다.


In [ ]:
# 이 셀은 라이브러리 import, 업로드 이름 검사, 안전한 ZIP 해제, 소스 고정을 담당합니다.
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
import csv
import hashlib
import html
import importlib.util
import io
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import unicodedata
import urllib.parse
import xml.etree.ElementTree as ET
import zipfile
from google.colab import files
from IPython.display import HTML, display

# 브라우저 업로드는 메모리도 사용하므로 제품 한도를 명시적으로 제한합니다.
MAX_FILENAME_BYTES = 240
MAX_MEDIA_FILES = 200
MAX_TIMELINE_ROWS = 5_000
MAX_SUBTITLE_ROWS = 10_000
MAX_SINGLE_MEDIA_BYTES = 1 * 1024**3
MAX_TOTAL_UPLOAD_BYTES = 2 * 1024**3
MAX_STARTER_FILES = 10_000
MAX_STARTER_UNCOMPRESSED_BYTES = 2 * 1024**3
MAX_STARTER_MEMBER_BYTES = 512 * 1024**2
MAX_COMPRESSION_RATIO = 1_000
VIDEO_SUFFIXES = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".heic", ".webp", ".bmp"}

def format_bytes(size):
    value = float(size)
    for unit in ("B", "KiB", "MiB", "GiB"):
        if value < 1024 or unit == "GiB":
            return f"{value:.1f} {unit}"
        value /= 1024

def sha256_file(path):
    # 소스 고정 검증과 결과 manifest가 같은 스트리밍 SHA-256 구현을 사용합니다.
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def has_control_character(text):
    # Cc=제어 문자, Cf=보이지 않는 형식 문자입니다. 경로 혼동을 막기 위해 파일명에서 거부합니다.
    return any(unicodedata.category(character) in {"Cc", "Cf"} for character in text)

def utf8_prefix(value, max_bytes):
    # 파일시스템 한계는 글자 수가 아니라 UTF-8 바이트 수이므로 문자 경계에서 안전하게 줄입니다.
    pieces = []
    used_bytes = 0
    for character in str(value):
        character_bytes = len(character.encode("utf-8"))
        if used_bytes + character_bytes > max_bytes:
            break
        pieces.append(character)
        used_bytes += character_bytes
    return "".join(pieces)

def safe_uploaded_basename(raw_name, *, label):
    # 업로드는 한 폴더에만 저장합니다. 슬래시·역슬래시·상위 경로는 허용하지 않습니다.
    name = str(raw_name)
    if not name or name in {".", ".."}:
        raise ValueError(f"{label} 이름이 비어 있거나 안전하지 않습니다: {name!r}")
    if "/" in name or "\\" in name or Path(name).name != name:
        raise ValueError(f"{label}에는 폴더 경로를 넣을 수 없습니다: {name!r}")
    if has_control_character(name):
        raise ValueError(f"{label}에 제어 문자 또는 보이지 않는 문자가 있습니다: {name!r}")
    if len(name.encode("utf-8")) > MAX_FILENAME_BYTES:
        raise ValueError(f"{label} 이름이 너무 깁니다(UTF-8 기준 최대 {MAX_FILENAME_BYTES}바이트): {name!r}")
    return name

def portable_name_key(name):
    # Mac·Linux 사이의 Unicode 조합 차이와 대소문자 차이를 같은 이름으로 비교합니다.
    return unicodedata.normalize("NFC", name).casefold()

@contextmanager
def temporary_upload_directory():
    # files.upload()가 현재 폴더에 복사본을 남길 수 있어, 별도 임시 폴더에서 호출한 뒤 즉시 치웁니다.
    incoming = Path(tempfile.mkdtemp(prefix="incoming_", dir="/content"))
    previous = Path.cwd()
    try:
        os.chdir(incoming)
        yield
    finally:
        os.chdir(previous)
        shutil.rmtree(incoming, ignore_errors=True)

def direct_upload(prompt):
    print(prompt)
    with temporary_upload_directory():
        uploaded = dict(files.upload())
    if not uploaded:
        raise ValueError("선택한 파일이 없습니다. 이 셀을 다시 실행해 주세요.")
    return uploaded

def save_uploaded_bytes(payload, destination, *, replace=False):
    destination.parent.mkdir(parents=True, exist_ok=True)
    # 기획표 셀을 수정해 다시 실행할 때만 replace=True를 사용합니다. 그 외에는 기존 파일 덮어쓰기를 거부합니다.
    with destination.open("wb" if replace else "xb") as handle:
        handle.write(bytes(payload))
    destination.chmod(0o600)

def safe_extract_starter_zip(zip_path, destination):
    # 개발자 fallback ZIP만 다룹니다. 경로 탈출·심볼릭 링크·압축 폭탄을 먼저 거부합니다.
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(zip_path) as archive:
        infos = archive.infolist()
        if len(infos) > MAX_STARTER_FILES:
            raise ValueError(f"Starter ZIP 파일 수가 너무 많습니다: {len(infos)}")
        if sum(item.file_size for item in infos) > MAX_STARTER_UNCOMPRESSED_BYTES:
            raise ValueError("Starter ZIP의 압축 해제 크기가 제품 한도를 넘습니다.")
        for item in infos:
            name = item.filename
            member = PurePosixPath(name)
            unix_type = (item.external_attr >> 16) & 0o170000
            target = (destination / Path(*member.parts)).resolve()
            if item.flag_bits & 0x1:
                raise ValueError(f"암호화 ZIP 항목은 허용하지 않습니다: {name}")
            if not name or name.startswith(("/", "\\")) or "\\" in name or has_control_character(name):
                raise ValueError(f"안전하지 않은 Starter ZIP 이름입니다: {name!r}")
            if member.is_absolute() or ".." in member.parts:
                raise ValueError(f"상위 경로를 포함한 Starter ZIP 항목입니다: {name}")
            if target != destination and destination not in target.parents:
                raise ValueError(f"Starter ZIP 경로가 작업 폴더를 벗어납니다: {name}")
            if unix_type == stat.S_IFLNK:
                raise ValueError(f"Starter ZIP 심볼릭 링크는 허용하지 않습니다: {name}")
            ratio = item.file_size / max(1, item.compress_size)
            if item.file_size > MAX_STARTER_MEMBER_BYTES or ratio > MAX_COMPRESSION_RATIO:
                raise ValueError(f"비정상적으로 큰 Starter ZIP 항목입니다: {name}")
        archive.extractall(destination)
    return destination

def require_regular_file_inside(path, root, *, label):
    root = Path(root).resolve()
    path = Path(path)
    resolved = path.resolve()
    if path.is_symlink() or not resolved.is_file() or (resolved != root and root not in resolved.parents):
        raise ValueError(f"{label} 위치가 안전하지 않습니다: {path}")
    return resolved

# 설정값 자체도 코드 실행 전에 검사합니다. 프로젝트명은 표시용이며 경로로 직접 사용하지 않습니다.
PROJECT_NAME = str(PROJECT_NAME).strip()
if not PROJECT_NAME or len(PROJECT_NAME) > 100 or has_control_character(PROJECT_NAME):
    raise ValueError("PROJECT_NAME은 제어 문자 없는 1~100자 이름이어야 합니다.")
if OUTPUT_FORMAT not in {"portrait", "landscape", "both"}:
    raise ValueError("OUTPUT_FORMAT은 portrait, landscape, both 중 하나여야 합니다.")
if str(FPS) not in {"23.976", "24", "25", "29.97", "30", "50", "59.94", "60"}:
    raise ValueError("지원 FPS 목록에서 선택해 주세요.")
if SUBTITLE_MODE not in {"title", "caption", "both", "off"}:
    raise ValueError("SUBTITLE_MODE은 title, caption, both, off 중 하나여야 합니다.")
if not (0.1 <= float(DEFAULT_PHOTO_DURATION) <= 3600):
    raise ValueError("DEFAULT_PHOTO_DURATION은 0.1~3600초 사이여야 합니다.")

WORKSPACE = Path(tempfile.mkdtemp(prefix="fcpxml_safe_quick_", dir="/content"))
PROJECT_ROOT = WORKSPACE / "project"
MEDIA_DIR = PROJECT_ROOT / "Media"
PROJECT_ROOT.mkdir(mode=0o700)
MEDIA_DIR.mkdir(mode=0o700)
SOURCE_COMMIT = None

if str(REPO_URL).strip():
    # 공개 GitHub HTTPS 주소만 허용합니다. 쉘 문자열 결합 없이 인자 배열로 실행합니다.
    repository_url = str(REPO_URL).strip().rstrip("/")
    if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?", repository_url):
        raise ValueError("REPO_URL은 공개 GitHub HTTPS 저장소 주소여야 합니다.")
    repository_ref = str(REPO_REF).strip()
    if (not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._/-]{0,199}", repository_ref)
            or ".." in repository_ref or repository_ref.endswith((".", "/"))):
        raise ValueError("REPO_REF는 릴리스 태그·브랜치·고정 commit ref 형식이어야 합니다.")
    REPO_DIR = WORKSPACE / "repo"
    subprocess.run(["git", "init", "--quiet", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "remote", "add", "origin", repository_url], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "--depth", "1", "origin", repository_ref], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)
    SOURCE_COMMIT = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    SOURCE_MODE = "public_github_ref"
    if not SOURCE_HASHES_READY:
        raise RuntimeError(
            "공개 배포용 SOURCE_SHA256이 아직 확정되지 않았습니다. "
            "배포자가 릴리스 소스 해시를 채운 뒤 SOURCE_HASHES_READY=True로 바꿔야 합니다."
        )
    for relative_name, expected_sha256 in SOURCE_SHA256.items():
        if not re.fullmatch(r"[0-9a-f]{64}", str(expected_sha256)):
            raise RuntimeError(f"유효하지 않은 고정 SHA-256 값입니다: {relative_name}")
        source_path = require_regular_file_inside(
            REPO_DIR / Path(relative_name), REPO_DIR, label=f"고정 소스 {relative_name}"
        )
        actual_sha256 = sha256_file(source_path)
        if actual_sha256 != expected_sha256:
            raise RuntimeError(
                f"공개 소스 SHA-256 불일치: {relative_name}\n"
                f"기대: {expected_sha256}\n실제: {actual_sha256}"
            )
else:
    print("[개발자 전용 fallback] 검토할 CSV_to_FCPXML Starter ZIP 하나를 올리세요.")
    starter_upload = direct_upload("Starter ZIP 선택")
    if len(starter_upload) != 1:
        raise ValueError("개발자 fallback에서는 Starter ZIP을 정확히 1개만 올려야 합니다.")
    starter_name, starter_payload = next(iter(starter_upload.items()))
    safe_uploaded_basename(starter_name, label="Starter ZIP")
    if Path(starter_name).suffix.lower() != ".zip":
        raise ValueError("개발자 fallback 파일은 .zip이어야 합니다.")
    starter_zip = WORKSPACE / "developer_starter.zip"
    save_uploaded_bytes(starter_payload, starter_zip)
    starter_root = safe_extract_starter_zip(starter_zip, WORKSPACE / "starter_source")
    candidates = [
        path.parent for path in starter_root.rglob("make_xml.py")
        if "__MACOSX" not in path.parts
        and (path.parent / "csv_to_fcpxml.py").is_file()
        and (path.parent / "make_xml_input.py").is_file()
        and (path.parent / "spreadsheet_input.py").is_file()
        and (path.parent / "preview_report.py").is_file()
        and (path.parent / "scripts" / "validate_fcpxml.py").is_file()
    ]
    if len(candidates) != 1:
        raise ValueError(f"Starter 소스 루트를 하나로 결정할 수 없습니다: {len(candidates)}개")
    REPO_DIR = candidates[0].resolve()
    SOURCE_MODE = "developer_starter_zip"

# 검증한 실행 파일만 깨끗한 폴더로 복사합니다. 저장소의 다른 .py가 csv·argparse 같은
# 표준 모듈 이름을 가로채는 일을 막고, 실제 실행 범위를 일곱 파일로 한정합니다.
EXECUTION_SOURCE_FILES = (
    "make_xml.py", "make_xml_input.py", "csv_to_fcpxml.py",
    "simple_timeline.py",
    "spreadsheet_input.py",
    "preview_report.py",
    "scripts/validate_fcpxml.py",
)
if SOURCE_MODE == "public_github_ref" and set(SOURCE_SHA256) != set(EXECUTION_SOURCE_FILES):
    raise RuntimeError("공개 소스 해시 목록과 실제 실행 파일 목록이 일치하지 않습니다.")
SOURCE_ROOT = REPO_DIR
EXECUTION_DIR = WORKSPACE / "verified_execution"
EXECUTION_DIR.mkdir(mode=0o700)
for relative_name in EXECUTION_SOURCE_FILES:
    source_path = require_regular_file_inside(
        SOURCE_ROOT / Path(relative_name), SOURCE_ROOT, label=f"실행 소스 {relative_name}"
    )
    destination = EXECUTION_DIR / Path(relative_name)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source_path, destination)
    destination.chmod(0o500)
# 기획표 어댑터도 검증·복사된 실행 폴더의 정확한 파일에서만 불러옵니다.
SPREADSHEET_INPUT_PATH = require_regular_file_inside(
    EXECUTION_DIR / "spreadsheet_input.py", EXECUTION_DIR, label="spreadsheet_input.py"
)
spreadsheet_spec = importlib.util.spec_from_file_location(
    "verified_spreadsheet_input", SPREADSHEET_INPUT_PATH
)
if spreadsheet_spec is None or spreadsheet_spec.loader is None:
    raise RuntimeError("검증된 기획표 입력 모듈을 불러올 수 없습니다.")
spreadsheet_module = importlib.util.module_from_spec(spreadsheet_spec)
spreadsheet_spec.loader.exec_module(spreadsheet_module)
PlanInputError = spreadsheet_module.PlanInputError
decode_uploaded_plan = spreadsheet_module.decode_uploaded_plan
REPO_DIR = EXECUTION_DIR
MAKE_XML = require_regular_file_inside(REPO_DIR / "make_xml.py", REPO_DIR, label="make_xml.py")
VALIDATOR = require_regular_file_inside(REPO_DIR / "scripts" / "validate_fcpxml.py", REPO_DIR, label="FCPXML validator")
if shutil.which("ffprobe") is None or shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
print(f"준비 완료 · 소스 방식: {SOURCE_MODE} · commit: {SOURCE_COMMIT or 'Starter ZIP'}")


## 2. 타임라인 기획표 정확히 1개 업로드

여기에는 사진이나 영상이 아니라, 위 6개 기본 열로 작성한 `.csv` 또는 `.xlsx` 기획표 하나만 선택합니다. `순서`, `화면 맞춤`, `메모` 열을 추가한 확장 양식도 사용할 수 있습니다. Numbers 사용자는 `파일 → 다음으로 내보내기 → Excel`로 `.xlsx`를 만든 뒤 선택하세요.


In [ ]:
# 기본 양식은 6열입니다. 순서·화면 맞춤·메모는 필요한 사람만 추가하는 선택 열입니다.
DEFAULT_TIMELINE_HEADERS = [
    "파일", "영상 원본 시작", "영상 원본 끝",
    "사진 표시 시간(초)", "화면 자막", "소리",
]
EXTENDED_TIMELINE_HEADERS = [
    "순서", "파일", "영상 원본 시작", "영상 원본 끝",
    "사진 표시 시간(초)", "화면 자막", "소리", "화면 맞춤", "메모",
]
KNOWN_TIMELINE_HEADERS = set(EXTENDED_TIMELINE_HEADERS)
REQUIRED_TIMELINE_HEADERS = {"파일"}

def validate_timeline_headers(fieldnames):
    # 선택 열은 어떤 조합·순서로 넣어도 되지만, 파일 열은 반드시 하나만 있어야 합니다.
    actual_headers = [str(name or "").strip() for name in (fieldnames or [])]
    if not actual_headers:
        raise ValueError("타임라인 기획표 첫 줄에 열 이름이 없습니다.")
    duplicate_headers = sorted({name for name in actual_headers if actual_headers.count(name) > 1})
    if duplicate_headers:
        raise ValueError(f"타임라인 기획표에 같은 열 이름이 두 번 있습니다: {duplicate_headers}")
    if any("?" in name or "\ufffd" in name for name in actual_headers):
        raise ValueError(
            "타임라인 기획표 첫 줄의 한글이 깨진 것으로 보입니다. "
            "깨진 상태로 저장하지 말고 새 양식을 받은 뒤 `.xlsx` 또는 CSV UTF-8로 다시 저장해 주세요."
        )
    unknown_headers = sorted(set(actual_headers) - KNOWN_TIMELINE_HEADERS)
    if unknown_headers:
        raise ValueError(
            f"알 수 없는 타임라인 기획표 열입니다: {unknown_headers}\n"
            f"사용 가능한 열: {EXTENDED_TIMELINE_HEADERS}\n"
            "Excel에서 한글이 깨져 보였다면 새 양식을 받은 뒤 `.xlsx` 또는 CSV UTF-8로 다시 저장하세요."
        )
    missing_headers = sorted(REQUIRED_TIMELINE_HEADERS - set(actual_headers))
    if missing_headers:
        raise ValueError(f"타임라인 기획표에 꼭 필요한 열이 없습니다: {missing_headers}")
    return actual_headers
timeline_upload = direct_upload("타임라인 기획표(.csv 또는 .xlsx) 하나를 선택하세요.")
if len(timeline_upload) != 1:
    raise ValueError("타임라인 기획표는 정확히 1개만 올려야 합니다.")
timeline_original_name, timeline_payload = next(iter(timeline_upload.items()))
safe_uploaded_basename(timeline_original_name, label="타임라인 기획표")
try:
    timeline_text, timeline_canonical_bytes, timeline_source_description = decode_uploaded_plan(
        timeline_original_name, timeline_payload, kind="timeline", max_rows=MAX_TIMELINE_ROWS,
    )
except PlanInputError as error:
    raise ValueError(str(error)) from None
if "\x00" in timeline_text:
    raise ValueError("타임라인 기획표에 NUL 제어 문자가 있습니다.")
timeline_reader = csv.DictReader(io.StringIO(timeline_text, newline=""))
raw_headers = list(timeline_reader.fieldnames or [])
actual_headers = validate_timeline_headers(raw_headers)
has_order_column = "순서" in actual_headers
TIMELINE_ROWS = []
seen_orders = set()
for row_number, raw_row in enumerate(timeline_reader, start=2):
    if not any(str(value or "").strip() for value in raw_row.values()):
        continue
    if None in raw_row:
        raise ValueError(f"타임라인 기획표 {row_number}행 값이 헤더보다 많습니다. 쉼표와 따옴표를 확인하세요.")
    # 헤더 앞뒤 공백은 정리하되, 알 수 없는 열이나 중복 열은 위에서 이미 거부했습니다.
    row = {str(key or "").strip(): value for key, value in raw_row.items()}
    if has_order_column:
        order_text = str(row.get("순서", "")).strip()
        if not order_text.isdigit() or int(order_text) < 1 or int(order_text) in seen_orders:
            raise ValueError(f"타임라인 기획표 {row_number}행 순서는 중복 없는 1 이상의 정수여야 합니다: {order_text!r}")
        seen_orders.add(int(order_text))
    media_name = safe_uploaded_basename(str(row.get("파일", "")).strip(), label=f"기획표 {row_number}행 파일")
    row["파일"] = media_name
    TIMELINE_ROWS.append(row)
    if len(TIMELINE_ROWS) > MAX_TIMELINE_ROWS:
        raise ValueError(f"타임라인은 최대 {MAX_TIMELINE_ROWS}개 장면까지 지원합니다.")
if not TIMELINE_ROWS:
    raise ValueError("타임라인 기획표에 장면 행이 없습니다.")
TIMELINE_PATH = PROJECT_ROOT / "timeline.csv"
save_uploaded_bytes(timeline_canonical_bytes, TIMELINE_PATH, replace=True)
order_description = "순서 열 기준" if has_order_column else "기획표 행 순서 기준"
print(f"타임라인 준비 완료: {len(TIMELINE_ROWS)}개 장면 · {order_description} · 원본 {timeline_original_name} · {timeline_source_description}")
display_rows = sorted(TIMELINE_ROWS, key=lambda item: int(str(item["순서"]).strip())) if has_order_column else TIMELINE_ROWS
for position, row in enumerate(display_rows[:10], start=1):
    order_label = row["순서"] if has_order_column else position
    print(f"  {order_label}. {row['파일']}")
if len(TIMELINE_ROWS) > 10:
    print(f"  …외 {len(TIMELINE_ROWS) - 10}개")


## 3. 선택 사항: 정밀 자막 기획표 정확히 1개 업로드

`USE_SEPARATE_SUBTITLES_CSV=False`이면 업로드 창이 열리지 않고 이 단계를 건너뜁니다. 한 장면에 자막이 여러 번 바뀌는 경우에만 설정을 켜고 정밀 자막 `.csv` 또는 `.xlsx` 기획표 하나를 올리세요.


In [ ]:
# 별도 자막 사용을 명시한 경우에만 업로드를 요청합니다.
EXPECTED_SUBTITLE_HEADERS = ["번호", "시작", "끝", "최종 대사", "장면 의도"]
SUBTITLES_PATH = None
SUBTITLE_BYTES = 0
subtitle_rows = []
if USE_SEPARATE_SUBTITLES_CSV:
    subtitle_upload = direct_upload("정밀 자막 기획표(.csv 또는 .xlsx) 하나를 선택하세요.")
    if len(subtitle_upload) != 1:
        raise ValueError("정밀 자막 기획표는 정확히 1개만 올려야 합니다.")
    subtitle_original_name, subtitle_payload = next(iter(subtitle_upload.items()))
    safe_uploaded_basename(subtitle_original_name, label="정밀 자막 기획표")
    try:
        subtitle_text, subtitle_canonical_bytes, subtitle_source_description = decode_uploaded_plan(
            subtitle_original_name, subtitle_payload, kind="subtitles", max_rows=MAX_SUBTITLE_ROWS,
        )
    except PlanInputError as error:
        raise ValueError(str(error)) from None
    subtitle_reader = csv.DictReader(io.StringIO(subtitle_text, newline=""))
    subtitle_headers = [str(name or "").strip() for name in (subtitle_reader.fieldnames or [])]
    if subtitle_headers != EXPECTED_SUBTITLE_HEADERS:
        raise ValueError(
            f"정밀 자막 기획표 첫 줄이 다릅니다. 필요: {EXPECTED_SUBTITLE_HEADERS} / 현재: {subtitle_headers}. "
            "한글이 깨져 보이면 새 양식을 받은 뒤 `.xlsx` 또는 CSV UTF-8로 다시 저장하세요."
        )
    subtitle_rows = [row for row in subtitle_reader if any(str(value or "").strip() for value in row.values())]
    if not subtitle_rows:
        raise ValueError("정밀 자막 기획표에 자막 행이 없습니다.")
    if len(subtitle_rows) > MAX_SUBTITLE_ROWS:
        raise ValueError(f"정밀 자막은 최대 {MAX_SUBTITLE_ROWS}행까지 지원합니다.")
    if any(str(row.get("화면 자막", "")).strip() for row in TIMELINE_ROWS):
        raise ValueError(
            "자막 입력이 두 곳에 있습니다. 별도 자막 기획표를 사용하려면 "
            "타임라인 기획표의 '화면 자막' 열을 모두 비운 뒤 2단계부터 다시 실행하세요."
        )
    SUBTITLES_PATH = PROJECT_ROOT / "subtitles.csv"
    save_uploaded_bytes(subtitle_canonical_bytes, SUBTITLES_PATH, replace=True)
    SUBTITLE_BYTES = len(subtitle_payload)
    print(f"정밀 자막 준비 완료: {len(subtitle_rows)}개 · 원본 {subtitle_original_name} · {subtitle_source_description}")
else:
    # 앞 실행에서 별도 자막을 사용했더라도 설정을 끄면 stale 파일을 남기지 않습니다.
    (PROJECT_ROOT / "subtitles.csv").unlink(missing_ok=True)
    print("정밀 자막 기획표를 사용하지 않습니다. 타임라인 기획표의 화면 자막 열을 사용합니다.")


## 4. 기획표 내용 확인 후 사진·영상 파일 업로드

먼저 표에서 파일명·사용 구간·자막 문구와 줄바꿈이 정상인지 확인합니다. 이 표는 문구 확인용이며, 정확한 폰트·크기·위치·윤곽선은 생성된 `*_with_titles.fcpxml`을 Final Cut Pro로 가져온 뒤 확인합니다. 정상일 때만 `OK`를 입력하면 미디어 선택 창이 열립니다.

그다음 타임라인 기획표의 `파일` 열에 적은 원본을 모두 선택합니다. 사진과 영상을 섞어도 되고 `종류`를 따로 입력하지 않습니다. 업로드 후 파일 종류·크기·영상 길이·소리 유무를 자동으로 보여주며, 기획표와 이름이 정확히 맞지 않으면 XML 생성 전에 수정 방법을 알려줍니다.


In [ ]:
# 기획표 미리보기는 미디어를 올리기 전에 한글 깨짐, 파일명, 시간과 자막 문구를 사람이 확인하게 합니다.
PREVIEW_MAX_ROWS = 20
PREVIEW_MAX_CHARS = 300

def preview_cell(value):
    # 사용자 기획표 문자열은 HTML로 해석되지 않도록 반드시 escape합니다. 줄바꿈만 화면에 유지합니다.
    text = str(value or "").strip()
    if not text:
        return '<span style="color:#6b7280">(비어 있음)</span>'
    if len(text) > PREVIEW_MAX_CHARS:
        text = text[:PREVIEW_MAX_CHARS] + "…"
    return html.escape(text).replace("\n", "<br>")

def display_plan_preview(title, headers, rows):
    # 큰 기획표는 앞부분만 그려 브라우저가 멈추지 않게 하고, 실제 생성에는 모든 행을 사용합니다.
    visible_rows = rows[:PREVIEW_MAX_ROWS]
    header_html = "".join(
        f'<th style="padding:8px;text-align:left;background:#f3f4f6;position:sticky;top:0">{html.escape(header)}</th>'
        for header in headers
    )
    body_html = "".join(
        "<tr>"
        + "".join(
            f'<td style="padding:8px;border-top:1px solid #e5e7eb;vertical-align:top">{preview_cell(value)}</td>'
            for value in row
        )
        + "</tr>"
        for row in visible_rows
    )
    hidden_count = len(rows) - len(visible_rows)
    remainder = (
        f"<p>앞 {len(visible_rows)}행만 표시했습니다. 나머지 {hidden_count}행도 생성에는 모두 사용됩니다.</p>"
        if hidden_count else ""
    )
    display(HTML(
        f"<h3>{html.escape(title)}</h3>"
        '<div style="max-height:420px;overflow:auto;border:1px solid #d1d5db">'
        '<table style="border-collapse:collapse;width:100%;font-size:14px">'
        f"<thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody>"
        "</table></div>" + remainder
    ))

def describe_timeline_range(row):
    suffix = Path(row["파일"]).suffix.lower()
    if suffix in IMAGE_SUFFIXES:
        duration = str(row.get("사진 표시 시간(초)", "") or "").strip()
        return f"사진 {duration or DEFAULT_PHOTO_DURATION}초" + (" (기본)" if not duration else "")
    if suffix in VIDEO_SUFFIXES:
        start = str(row.get("영상 원본 시작", "") or "").strip()
        end = str(row.get("영상 원본 끝", "") or "").strip()
        if start or end:
            return f"{start or '(시작 없음)'} → {end or '(끝 없음)'}"
        return "영상 전체"
    return "확장자 확인 필요"

timeline_preview_rows = [
    (
        row["순서"] if has_order_column else position,
        row["파일"],
        describe_timeline_range(row),
        row.get("화면 자막", ""),
    )
    for position, row in enumerate(display_rows, start=1)
]
display_plan_preview(
    "타임라인 기획표 확인",
    ["순서", "파일", "사용 구간 / 표시 시간", "화면 자막"],
    timeline_preview_rows,
)

if USE_SEPARATE_SUBTITLES_CSV:
    subtitle_preview_rows = [
        (
            row.get("번호", ""),
            f"{row.get('시작', '')} → {row.get('끝', '')}",
            row.get("최종 대사", ""),
        )
        for row in subtitle_rows
    ]
    display_plan_preview(
        "정밀 자막 기획표 확인",
        ["번호", "시작 → 끝", "최종 대사"],
        subtitle_preview_rows,
    )

subtitle_texts = (
    [str(row.get("최종 대사", "") or "") for row in subtitle_rows]
    if USE_SEPARATE_SUBTITLES_CSV
    else [str(row.get("화면 자막", "") or "") for row in display_rows]
)
if any("\ufffd" in text or text.count("?") >= 3 for text in subtitle_texts):
    print("⚠️ 자막에 깨진 글자로 의심되는 문자가 있습니다. 표를 특히 주의해서 확인하세요.")

display(HTML(
    '<div style="padding:12px;border-left:4px solid #f59e0b;background:#fffbeb">'
    "<strong>확인 범위</strong><br>"
    "이 표는 한글 깨짐, 파일명, 시간, 줄바꿈과 자막 문구를 확인하기 위한 것입니다. "
    "정확한 폰트·크기·위치·윤곽선은 <code>*_with_titles.fcpxml</code>을 "
    "Final Cut Pro로 가져온 뒤 확인합니다.</div>"
))
preview_confirmation = input(
    "표의 한글과 자막이 정상이라면 OK를 입력하세요. 깨졌다면 기획표를 다시 저장한 뒤 올려주세요: "
).strip().upper()
if preview_confirmation != "OK":
    raise ValueError(
        "기획표 확인이 완료되지 않아 미디어 업로드를 시작하지 않았습니다. "
        "기획표를 수정한 뒤 2단계부터 다시 실행하세요."
    )
print("기획표 확인 완료. 이제 사진·영상 파일을 선택하세요.")

# 미디어는 파일명 안전성, NFC+casefold 중복, 개수·개별 크기·전체 크기를 검사한 뒤 Media/에 저장합니다.
media_upload = direct_upload("기획표에 적은 사진·영상 파일을 모두 선택하세요.")
if len(media_upload) > MAX_MEDIA_FILES:
    raise ValueError(f"미디어는 한 작업에 최대 {MAX_MEDIA_FILES}개입니다: {len(media_upload)}개")
media_total_bytes = sum(len(payload) for payload in media_upload.values())
all_input_bytes = len(timeline_payload) + SUBTITLE_BYTES + media_total_bytes
if all_input_bytes > MAX_TOTAL_UPLOAD_BYTES:
    raise ValueError(f"전체 직접 업로드 한도는 2 GiB입니다: {format_bytes(all_input_bytes)}")
if all_input_bytes + 256 * 1024**2 > shutil.disk_usage(WORKSPACE).free:
    raise ValueError("Colab 임시 디스크 여유 공간이 부족합니다. 파일 수나 크기를 줄여 주세요.")

media_by_exact_name = {}
media_by_portable_key = {}
for raw_name, payload in media_upload.items():
    name = safe_uploaded_basename(raw_name, label="미디어 파일")
    suffix = Path(name).suffix.lower()
    if suffix not in VIDEO_SUFFIXES | IMAGE_SUFFIXES:
        raise ValueError(f"지원하지 않는 사진·영상 확장자입니다: {name}")
    if len(payload) > MAX_SINGLE_MEDIA_BYTES:
        raise ValueError(f"미디어 한 파일 한도는 1 GiB입니다: {name} ({format_bytes(len(payload))})")
    key = portable_name_key(name)
    if key in media_by_portable_key:
        raise ValueError(
            "대소문자 또는 Unicode 표기만 다른 중복 미디어입니다: "
            f"{media_by_portable_key[key]!r}, {name!r}"
        )
    media_by_portable_key[key] = name
    media_by_exact_name[name] = payload

# 기획표 이름은 추측해서 바꾸지 않습니다. 거의 같은 이름이 있으면 정확한 두 이름을 함께 안내합니다.
missing_names = []
for row_number, row in enumerate(TIMELINE_ROWS, start=2):
    requested = row["파일"]
    if requested in media_by_exact_name:
        continue
    similar = media_by_portable_key.get(portable_name_key(requested))
    if similar is not None:
        raise ValueError(
            f"기획표 {row_number}행 파일명이 업로드 이름과 정확히 다릅니다. "
            f"기획표: {requested!r} / 업로드: {similar!r}"
        )
    missing_names.append(f"{row_number}행 {requested}")
if missing_names:
    raise ValueError("기획표에 있지만 업로드되지 않은 미디어:\n- " + "\n- ".join(missing_names))

# 이 셀을 다시 실행할 수 있도록 검사가 모두 끝난 뒤 기존 임시 Media/만 새로 만듭니다.
shutil.rmtree(MEDIA_DIR)
MEDIA_DIR.mkdir(mode=0o700)
for name, payload in media_by_exact_name.items():
    save_uploaded_bytes(payload, MEDIA_DIR / name)

def inspect_uploaded_media(path):
    # 확장자로 사진/영상 후보를 정하고 ffprobe로 실제 영상 스트림·크기·오디오를 확인합니다.
    command = [
        "ffprobe", "-v", "error", "-protocol_whitelist", "file,pipe", "-show_entries",
        "format=duration:stream=codec_type,width,height", "-of", "json", str(path),
    ]
    try:
        result = subprocess.run(command, text=True, capture_output=True, timeout=60)
    except subprocess.TimeoutExpired as error:
        raise ValueError(
            f"미디어 검사 시간이 60초를 넘었습니다: {path.name}. "
            "파일이 손상되지 않았는지 확인하거나 더 작은 호환 파일로 변환해 주세요."
        ) from error
    if result.returncode != 0:
        detail = result.stderr.strip() or "파일을 읽을 수 없습니다."
        raise ValueError(f"미디어 검사 실패: {path.name}\n{detail}")
    payload = json.loads(result.stdout)
    streams = payload.get("streams", [])
    video_stream = next((stream for stream in streams if stream.get("codec_type") == "video"), None)
    if video_stream is None:
        raise ValueError(f"사진 또는 영상 스트림을 찾을 수 없습니다: {path.name}")
    kind = "사진" if path.suffix.lower() in IMAGE_SUFFIXES else "영상"
    resolution = f"{video_stream.get('width', '?')}×{video_stream.get('height', '?')}"
    has_audio = any(stream.get("codec_type") == "audio" for stream in streams)
    duration = payload.get("format", {}).get("duration")
    duration_text = f"{float(duration):.2f}초" if kind == "영상" and duration not in {None, "N/A"} else "-"
    return kind, resolution, duration_text, ("있음" if has_audio else "없음")

print("미디어 분석 결과")
print("종류 | 파일 | 해상도 | 길이 | 원본 소리 | 크기")
for name in sorted(media_by_exact_name, key=portable_name_key):
    kind, resolution, duration_text, audio_text = inspect_uploaded_media(MEDIA_DIR / name)
    print(f"{kind} | {name} | {resolution} | {duration_text} | {audio_text} | {format_bytes(len(media_by_exact_name[name]))}")
referenced_names = {row["파일"] for row in TIMELINE_ROWS}
unused_names = sorted(set(media_by_exact_name) - referenced_names, key=portable_name_key)
if unused_names:
    print("참고: 기획표에서 사용하지 않는 업로드 파일(결과에는 포함되지 않음):", ", ".join(unused_names))
print(f"입력 검사 1차 통과 · 장면 {len(TIMELINE_ROWS)}개 · 미디어 {len(media_by_exact_name)}개 · {format_bytes(all_input_bytes)}")


## 5. 입력 최종 검사 후 Final Cut 프로젝트 생성

이 셀은 먼저 검사 전용 실행을 하고, 통과했을 때만 실제 결과를 만듭니다. 사진은 필요한 표시 시간만큼 MP4 편집 캐시로 변환됩니다. `OUTPUT_FORMAT=both`이면 같은 입력으로 세로 9:16과 가로 16:9를 순차 생성합니다. `SUBTITLE_MODE`의 기본 `title`은 일반 Basic Title 결과이며, `caption`은 접근성 캡션, `both`는 타이틀·캡션 XML 둘 다, `off`는 XML 자막을 끄는 선택입니다.


In [ ]:
# 사용자용 진입점인 make_xml.py만 호출합니다. 내부 빌더 함수를 노트북에서 직접 조립하지 않습니다.
def run_checked(command, *, purpose):
    print(f"[{purpose}]")
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout.rstrip())
    if result.stderr:
        print(result.stderr.rstrip(), file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"{purpose}에 실패했습니다(exit {result.returncode}). 위 오류 행을 수정한 뒤 다시 실행하세요.")

TARGET_SPECS = {
    "portrait": {"directory": "vertical_9x16", "label": "세로 9:16", "width": 1080, "height": 1920},
    "landscape": {"directory": "horizontal_16x9", "label": "가로 16:9", "width": 1920, "height": 1080},
}
EXPECTED_FRAME_DURATIONS = {
    "23.976": "1001/24000s", "24": "1/24s", "25": "1/25s",
    "29.97": "1001/30000s", "30": "1/30s", "50": "1/50s",
    "59.94": "1001/60000s", "60": "1/60s",
}
selected_layouts = ["portrait", "landscape"] if OUTPUT_FORMAT == "both" else [OUTPUT_FORMAT]
expected_target_directories = {TARGET_SPECS[layout]["directory"] for layout in selected_layouts}
print("생성 대상:", " → ".join(TARGET_SPECS[layout]["label"] for layout in selected_layouts))
if OUTPUT_FORMAT == "both":
    print("한 번 업로드한 기획표와 미디어로 두 대상을 순서대로 생성합니다.")

GENERATED = PROJECT_ROOT / "Generated"
if GENERATED.exists():
    # 작업공간 안의 고정 결과 폴더만 제거합니다. 사용자가 올린 Media/는 건드리지 않습니다.
    shutil.rmtree(GENERATED)
base_command = [
    sys.executable, str(MAKE_XML), str(PROJECT_ROOT),
    "--timeline", "timeline.csv",
    "--media-dir", "Media",
    "--output-dir", "Generated",
    "--project-name", PROJECT_NAME,
    "--fps", str(FPS),
    "--photo-duration", str(DEFAULT_PHOTO_DURATION),
    "--subtitle-mode", SUBTITLE_MODE,
    "--layout", OUTPUT_FORMAT,
]
if SUBTITLES_PATH is not None:
    base_command += ["--subtitles", "subtitles.csv"]
if not EXPORT_SRT:
    base_command.append("--no-srt")
run_checked(base_command + ["--validate-only"], purpose="입력 최종 검사")
run_checked(base_command, purpose="FCPXML 생성")

xml_files = sorted(GENERATED.rglob("*.fcpxml"))
if not xml_files:
    raise RuntimeError("생성된 FCPXML이 없습니다.")
actual_target_directories = {path.relative_to(GENERATED).parts[0] for path in xml_files}
if actual_target_directories != expected_target_directories:
    raise RuntimeError(
        "방향별 결과 폴더가 예상과 다릅니다. "
        f"예상={sorted(expected_target_directories)}, 실제={sorted(actual_target_directories)}"
    )
run_checked(
    [sys.executable, str(VALIDATOR), "--check-media", "--require-relative",
     "--project-root", str(PROJECT_ROOT), *map(str, xml_files)],
    purpose="상대경로와 미디어 참조 검사",
)
for xml_path in xml_files:
    xml_text = xml_path.read_text(encoding="utf-8")
    for forbidden in (str(WORKSPACE), "/content/"):
        if forbidden in xml_text:
            raise RuntimeError(f"Colab 절대경로가 XML에 남았습니다: {xml_path.name}")
    root = ET.fromstring(xml_text)
    for media_rep in root.findall(".//media-rep"):
        src = media_rep.get("src", "")
        parsed = urllib.parse.urlparse(src)
        if parsed.scheme or src.startswith(("/", "\\")):
            raise RuntimeError(f"상대 URI가 아닌 미디어 참조입니다: {src}")

# manifest의 targets는 요청값만 복사하지 않고 실제 FCPXML 프로젝트 format에서 읽어 검증합니다.
manifest_targets = []
for layout in selected_layouts:
    spec = TARGET_SPECS[layout]
    target_dir = GENERATED / spec["directory"]
    target_xml_files = sorted(target_dir.rglob("*.fcpxml"))
    if not target_xml_files:
        raise RuntimeError(f"{spec['label']} FCPXML이 없습니다: {target_dir}")
    actual_formats = set()
    for xml_path in target_xml_files:
        root = ET.fromstring(xml_path.read_text(encoding="utf-8"))
        sequence = root.find(".//project/sequence")
        if sequence is None or not sequence.get("format"):
            raise RuntimeError(f"프로젝트 format 참조가 없는 FCPXML입니다: {xml_path}")
        formats = {item.get("id"): item for item in root.findall("./resources/format")}
        project_format = formats.get(sequence.get("format"))
        if project_format is None:
            raise RuntimeError(f"프로젝트 format 리소스를 찾을 수 없습니다: {xml_path}")
        try:
            width = int(project_format.get("width", ""))
            height = int(project_format.get("height", ""))
        except ValueError as error:
            raise RuntimeError(f"프로젝트 해상도를 읽을 수 없습니다: {xml_path}") from error
        frame_duration = project_format.get("frameDuration", "")
        actual_formats.add((width, height, frame_duration))
    if len(actual_formats) != 1:
        raise RuntimeError(f"같은 대상 폴더의 FCPXML format이 서로 다릅니다: {sorted(actual_formats)}")
    width, height, frame_duration = actual_formats.pop()
    if (width, height) != (spec["width"], spec["height"]):
        raise RuntimeError(
            f"{spec['label']} 해상도가 예상과 다릅니다: "
            f"예상 {spec['width']}×{spec['height']}, 실제 {width}×{height}"
        )
    expected_frame_duration = EXPECTED_FRAME_DURATIONS[str(FPS)]
    if frame_duration != expected_frame_duration:
        raise RuntimeError(
            f"{spec['label']} FPS가 예상과 다릅니다: "
            f"예상 frameDuration={expected_frame_duration}, 실제={frame_duration}"
        )
    manifest_targets.append({
        "layout": layout,
        "directory": (Path("Generated") / spec["directory"]).as_posix(),
        "width": width,
        "height": height,
        "fps": str(FPS),
        "frame_duration": frame_duration,
        "fcpxml_files": [
            (Path("Generated") / path.relative_to(GENERATED)).as_posix()
            for path in target_xml_files
        ],
    })
    print(
        f"검증 완료 · {spec['label']} · {width}×{height} · {FPS}fps · "
        f"FCPXML {len(target_xml_files)}개"
    )
print("빌드와 상대경로·해상도·FPS 검사 완료:", ", ".join(path.name for path in xml_files))


## 6. 허용된 결과만 ZIP으로 내려받기

결과 ZIP에는 방향별 `Generated/vertical_9x16/`, `Generated/horizontal_16x9/` 폴더 중 선택한 대상의 FCPXML·SRT·검사 보고서·JSON과, 사진이 있을 때 각 대상의 `.build_media/` 아래 MP4 캐시만 들어갑니다. Python·셸·앱·설치 파일과 업로드한 원본 영상은 넣지 않습니다. 다만 사진 캐시에는 사진 픽셀이, XML·SRT·보고서에는 파일명·자막·메모가 남을 수 있으므로 결과 ZIP도 원본처럼 민감하게 보관하세요. 내려받기 직전에 각 파일의 크기와 SHA-256을 표시하고, 실제 XML에서 확인한 방향·해상도·FPS를 `manifest.json`의 `targets`에 기록합니다.


In [ ]:
# 결과 허용 목록을 통과한 일반 파일만 패키징합니다. 확장자만 맞춘 실행 파일도 차단합니다.
ALLOWED_RESULT_SUFFIXES = {".fcpxml", ".srt", ".txt", ".json"}
EXECUTABLE_OR_SCRIPT_SUFFIXES = {
    ".app", ".bat", ".cmd", ".com", ".command", ".dll", ".dylib",
    ".exe", ".jar", ".js", ".msi", ".pkg", ".ps1", ".py", ".scr",
    ".sh", ".so", ".vbs",
}

result_paths = []
for path in sorted(GENERATED.rglob("*"), key=lambda item: item.as_posix()):
    relative = path.relative_to(GENERATED)
    if path.is_symlink():
        raise RuntimeError(f"결과 심볼릭 링크는 허용하지 않습니다: {relative}")
    if path.is_dir():
        continue
    if not path.is_file():
        raise RuntimeError(f"결과에 일반 파일이 아닌 항목이 있습니다: {relative}")
    if not relative.parts or relative.parts[0] not in expected_target_directories:
        raise RuntimeError(f"선택한 방향별 결과 폴더 밖의 파일은 허용하지 않습니다: {relative}")
    component_suffixes = {Path(part).suffix.lower() for part in relative.parts}
    if component_suffixes & EXECUTABLE_OR_SCRIPT_SUFFIXES or path.stat().st_mode & 0o111:
        raise RuntimeError(f"실행 가능하거나 스크립트인 결과는 허용하지 않습니다: {relative}")
    suffix = path.suffix.lower()
    if suffix == ".mp4":
        if len(relative.parts) < 3 or relative.parts[1] != ".build_media":
            raise RuntimeError(f"MP4는 방향별 .build_media/ 아래 사진 캐시만 허용합니다: {relative}")
    elif suffix not in ALLOWED_RESULT_SUFFIXES:
        raise RuntimeError(f"허용되지 않은 결과 확장자입니다: {relative}")
    result_paths.append(path)
if not any(path.suffix.lower() == ".fcpxml" for path in result_paths):
    raise RuntimeError("패키징할 FCPXML이 없습니다.")

manifest_files = []
for path in result_paths:
    relative_path = (Path("Generated") / path.relative_to(GENERATED)).as_posix()
    manifest_files.append({
        "relative_path": relative_path,
        "target": path.relative_to(GENERATED).parts[0],
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })
manifest = {
    "schema_version": 2,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "project_name": PROJECT_NAME,
    "settings": {
        "output_format": OUTPUT_FORMAT, "fps": str(FPS), "subtitle_mode": SUBTITLE_MODE,
        "export_srt": bool(EXPORT_SRT), "default_photo_duration_seconds": float(DEFAULT_PHOTO_DURATION),
    },
    "targets": manifest_targets,
    "source": {"mode": SOURCE_MODE, "repo_ref": str(REPO_REF), "resolved_commit": SOURCE_COMMIT},
    "files": manifest_files,
}
MANIFEST_PATH = WORKSPACE / "manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

safe_result_name = re.sub(r"[^0-9A-Za-z가-힣._-]+", "_", PROJECT_NAME).strip("._") or "FinalCut"
safe_result_name = utf8_prefix(safe_result_name, 180).rstrip("._-") or "FinalCut"
RESULT_ZIP = WORKSPACE / f"{safe_result_name}_FinalCut_Result.zip"
with zipfile.ZipFile(RESULT_ZIP, "w", compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for path, item in zip(result_paths, manifest_files):
        archive.write(path, item["relative_path"])
    archive.write(MANIFEST_PATH, "manifest.json")
with zipfile.ZipFile(RESULT_ZIP) as archive:
    if archive.testzip() is not None:
        raise RuntimeError("결과 ZIP 무결성 검사에 실패했습니다.")
    expected_names = {item["relative_path"] for item in manifest_files} | {"manifest.json"}
    if set(archive.namelist()) != expected_names:
        raise RuntimeError("결과 ZIP의 실제 파일 목록이 manifest 대상과 다릅니다.")

print("다운로드 파일 목록 · 크기 · SHA-256")
for item in manifest_files:
    print(f"{item['relative_path']} | {format_bytes(item['size_bytes'])} | {item['sha256']}")
print(f"manifest.json | {format_bytes(MANIFEST_PATH.stat().st_size)} | {sha256_file(MANIFEST_PATH)}")
print("manifest targets:", ", ".join(
    f"{item['directory']} ({item['width']}×{item['height']} · {item['fps']}fps)"
    for item in manifest_targets
))
print(f"결과 ZIP: {RESULT_ZIP.name} ({format_bytes(RESULT_ZIP.stat().st_size)})")
files.download(str(RESULT_ZIP))


## 7. Mac에서 Final Cut Pro로 가져오기

1. ZIP 다운로드가 끝났는지 확인하고 압축을 풉니다.
2. Mac에서 새 프로젝트 폴더를 만들고, Colab에 올렸던 **동일한 이름의 원본**을 `Media/`에 넣습니다.
3. 결과의 `Generated/`를 `Media/` 옆에 둡니다. 사진이 있다면 방향별 폴더 안의 숨김 폴더 `.build_media/`도 삭제하지 마세요.
4. Final Cut Pro에서 Library와 Event를 선택하고 `File → Import → XML`을 누릅니다.
5. 세로는 `vertical_9x16/`, 가로는 `horizontal_16x9/`에서 먼저 `*_with_titles.fcpxml`을 선택합니다. 없다면 `*_clean.fcpxml`을 선택합니다. `both` 결과라면 두 파일을 각각 한 번씩 가져옵니다.
6. 누락 미디어, 화면 비율, 자막 글꼴·위치, 원본 소리, 사진 길이를 확인한 뒤 평소처럼 Share로 내보냅니다.

폴더 구조는 다음과 같습니다.

```text
MyVideo/
├─ Media/                 ← Colab에 올렸던 원본 사진·영상
└─ Generated/             ← 결과 ZIP에서 꺼낸 폴더
   ├─ vertical_9x16/
   │  ├─ MyVideo_vertical_9x16_with_titles.fcpxml
   │  ├─ MyVideo_vertical_9x16_clean.fcpxml
   │  └─ .build_media/      ← 세로용 사진 캐시
   └─ horizontal_16x9/
      ├─ MyVideo_horizontal_16x9_with_titles.fcpxml
      ├─ MyVideo_horizontal_16x9_clean.fcpxml
      └─ .build_media/      ← 가로용 사진 캐시
```

FCPXML은 편집 가능한 러프컷 뼈대입니다. 전환, 색 보정, 음악 선택, 정교한 오디오 믹싱과 최종 검수는 Final Cut Pro에서 진행합니다. Mac을 계속 사용할 예정이라면 다음 작업부터는 저장소의 Mac 가이드를 따라 로컬 가상환경에서 실행하는 편이 업로드 대기 없이 빠르고 원본을 외부 런타임에 올리지 않아도 되어 권장됩니다.


## 8. 다운로드 후 임시 파일과 런타임 삭제

다운로드를 확인한 뒤 아래 선택 셀을 실행해 이 노트북이 만든 임시 폴더를 지울 수 있습니다. 마지막으로 Colab 메뉴에서 **런타임 → 연결 해제 및 런타임 삭제**를 선택하세요. 브라우저 탭만 닫는 것보다 명확합니다.


In [ ]:
# 다운로드를 확인한 뒤에만 True로 바꾸고 실행하세요. 기본값 False라 '모두 실행'에서도 결과를 지우지 않습니다.
DELETE_COLAB_TEMP_FILES = False  # @param {type:"boolean"}
if DELETE_COLAB_TEMP_FILES:
    shutil.rmtree(WORKSPACE, ignore_errors=True)
    print("이 노트북이 만든 임시 작업 폴더를 삭제했습니다. 이제 런타임도 삭제하세요.")
else:
    print("임시 파일을 유지했습니다. 다운로드 확인 후 True로 바꾸어 다시 실행하세요.")
